# 🧬 Connect AI — 장기 기억 학습 (Unsloth)
내 1인 기업 지식을 모델 **가중치에 체득**시킵니다. 위 메뉴 **런타임 → 모두 실행**만 누르면 됩니다 (무료 T4 GPU).
- 데이터셋: `jealong/jealong1` (단기 지식 → conversations Q&A)
- 베이스 모델: `unsloth/gemma-4-E2B-it`  ← *내가 쓰는 모델로 바꿔도 됩니다 (누적 학습)*
- 결과 모델: `jealong/marketing_jeayoung` (GGUF — Connect AI 내장 엔진에 바로 로드, LM Studio 불필요)
- 설정: rank 16/alpha 32 · dropout 0 · lr 0.0003 · steps 40 · seq 1024 · linear · 양자화 q4_k_m (데이터 4개)


In [9]:
%%capture
# 버전을 직접 고정하지 않는다 — Unsloth가 현재 Colab torch에 맞는 의존성(torchao·transformers 등)을 알아서 설치.
# (고정 레시피는 Colab torch가 바뀌면 register_constant/recompile_limit 같은 충돌이 연쇄로 난다)
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo


## 🔑 HuggingFace 로그인 (맨 먼저!)
아래 칸에 **write 토큰**을 붙여넣으세요. *비공개 데이터셋을 불러오고*, 학습된 모델을 *업로드*하는 데 둘 다 필요해요.


In [15]:
from huggingface_hub import notebook_login
notebook_login()


In [16]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, max_seq_length = 1024,
    load_in_4bit = True, full_finetuning = False,
)
print("✅ 베이스 모델 로딩 완료")


==((====))==  Unsloth 2026.6.6: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
# LoRA — 전체의 1% 미만만 학습(메모리↓, 페르소나·핵심지식엔 충분)
model = FastModel.get_peft_model(
    model, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, finetune_vision_layers=False,
    r = 16, lora_alpha = 32, lora_dropout = 0, bias = "none", random_state = 3407,
)


## 📦 단기 지식 데이터셋 불러오기 (conversations Q&A)
Connect AI 앱이 업로드한 데이터셋. 각 행 = `{conversations:[{user},{assistant}]}`


In [17]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
ds = load_dataset("jealong/jealong1", data_files="connect-ai-brain.jsonl", split="train", token=True)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
def fmt(ex):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>") for c in ex["conversations"]]
    return {"text": texts}
ds = ds.map(fmt, batched=True)
print("데이터 개수:", len(ds)); print(ds[0]["text"][:400])


connect-ai-brain.jsonl:   0%|          | 0.00/4.30k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

데이터 개수: 4
<|turn>user
2026년 6월 11일 사이클3에서 디자이너의 채널 로고·배너 디자인 컨셉 기획 업무는 무엇인가요?<turn|>
<|turn>model
[운영 2026-06-11 사이클#3] Designer: 채널 로고·배너 디자인 컨셉 3안 기획 (오늘업무_2026-06-11/디자인_가이드.md)<turn|>



In [18]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 40, learning_rate = 0.0003,
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.001,
        lr_scheduler_type = "linear", seed = 3407, report_to = "none",
    ),
)


num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

In [19]:
# 🎭 응답(assistant)만 학습 — 질문 패턴은 마스킹(효율↑·품질↑)
# ⚠️ 마커는 모델/버전마다 다름(<|turn> vs <start_of_turn>) → 실제 텍스트에서 자동 감지
from unsloth.chat_templates import train_on_responses_only
_t = ds[0]["text"]
_im = "<|turn>user\n" if "<|turn>user" in _t else "<start_of_turn>user\n"
_rm = "<|turn>model\n" if "<|turn>model" in _t else "<start_of_turn>model\n"
trainer = train_on_responses_only(trainer, instruction_part=_im, response_part=_rm)
print(f"✅ 마스킹 마커 자동감지: {_rm.strip()} — 학습 준비 완료")


num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


Map (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


Filter (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

✅ 마스킹 마커 자동감지: <|turn>model — 학습 준비 완료


In [20]:
trainer_stats = trainer.train()
print("🎉 학습 완료! 최종 loss:", round(trainer_stats.training_loss, 4))
print("💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4 | Num Epochs = 40 | Total steps = 40
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 25,337,856 of 5,148,515,872 (0.49% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.461562
2,3.461562
3,3.461562
4,3.461562
5,3.461562
6,3.010166
7,2.391057
8,1.912779
9,1.615904
10,1.428518


🎉 학습 완료! 최종 loss: 0.9674
💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.


## 🧪 학습된 모델 테스트 (업로드 전에 확인!)
내가 가르친 지식을 직접 물어보세요. 답에 그 내용이 나오면 학습 성공이에요. 질문은 자유롭게 바꿔도 됩니다.


In [21]:
from unsloth import FastModel
FastModel.for_inference(model)
def chat(prompt, max_tokens=220):
    msg = [{"role":"user","content":[{"type":"text","text":prompt}]}]
    inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to("cuda")
    if inp["input_ids"][0,0].item() == tokenizer.bos_token_id:
        inp["input_ids"] = inp["input_ids"][:,1:]; inp["attention_mask"] = inp["attention_mask"][:,1:]
    out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\u2753 {prompt}\n\U0001F4AC {ans}\n" + "\u2500"*58)

# 👇 내가 가르친 지식에 대해 물어보세요 (자유롭게 수정)
chat("내 사업/지식에 대해 아는 걸 알려줘")
chat("너는 무엇을 도와줄 수 있어?")


❓ 내 사업/지식에 대해 아는 걸 알려줘
💬 어떤 사업이나 지식에 대해 알고 싶으신가요? **구체적인 주제를 알려주시면** 제가 그 분야에 대해 제가 아는 정보를 최대한 자세하게 알려드리겠습니다.

예를 들어, 다음과 같이 질문해 주시면 좋습니다.

**사업 관련 질문 예시:**

* "온라인 쇼핑몰을 시작하는 데 필요한 핵심 단계와 고려 사항을 알려줘."
* "소규모 비즈니스를 운영할 때 가장 흔하게 겪는 문제점과 해결책은 뭐야?"
* "구독 모델(Subscription Model)을 성공적으로 구축하려면 어떤 지식이 필요해?"

**지식/학문 관련 질문 예시:**

* "인공지능(AI)의 최신 발전 동향과 미래 전망에 대해 알려줘."
* "재무 관리(Financial Management)의 기본 원리와 투자 전략을 설명해 줘."
* "마케팅(Marketing) 분야에서 가장 중요한 트렌드 세 가지를 알려줘."

**주제를 알려주시면 바로 답변
──────────────────────────────────────────────────────────
❓ 너는 무엇을 도와줄 수 있어?
💬 저는 다양한 방법으로 사용자와 대화하고 도움을 드릴 수 있습니다. 예를 들어, 다음과 같은 일들을 도와드릴 수 있습니다:

**1. 정보 검색 및 제공:**
* 특정 주제에 대한 사실이나 정보를 찾아 드립니다. (예: "기후 변화의 영향에 대해 알려줘.")
* 정의나 설명이 필요한 경우 명확하게 설명해 드립니다.

**2. 창의적인 작업 지원:**
* 아이디어 구상, 브레인스토밍을 도와드립니다. (예: "새로운 마케팅 아이디어 5가지 생각해 줘.")
* 이야기, 시, 코드 초안 등을 작성하는 데 도움을 드릴 수 있습니다.

**3. 학습 및 튜터링:**
* 새로운 개념을 이해하기 쉽도록 도와드립니다. (예: "양자 역학의 기본 원리를 초보자 눈높이로 설명해 줘.")
* 언어 학습 시 대화 파트너 역할을 해 드릴 수 있습니다.

**4. 일상적인 대화 및 지원:**

──────

## 💾 GGUF로 저장 (Connect AI 내장 엔진용)
테스트가 만족스러우면 업로드! (맨 앞에서 로그인했으니 바로 됩니다)


In [ ]:
# 메모리 정리(OOM 방지) — 학습기 메모리 해제 후 변환
import gc, torch
try:
    del trainer
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()
print("메모리 정리 완료 — GGUF 변환 시작")
# 내 모델 = 장기 기억. q4_k_m GGUF 로 저장 + HF 업로드
model.push_to_hub_gguf("jealong/marketing_jeayoung", tokenizer, quantization_method="q4_k_m", token=True)
print("✅ 완료! Connect AI 앱 → 🤖 내 AI 팀 → HuggingFace에서 받기 → \"jealong/marketing_jeayoung\" 검색해서 내려받으면 내 모델로 바로 사용 (LM Studio 불필요)")


메모리 정리 완료 — GGUF 변환 시작
Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/5.04k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_wbb15_mm/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]